In [0]:
%pip install pytest

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
test_code = """
import pytest
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

def test_silver_train_not_empty():
    df = spark.table("databricks_gsales.silver.silver_train")
    assert df.count() > 0

def test_silver_train_no_nulls():
    df = spark.table("databricks_gsales.silver.silver_train")

    null_count = df.filter(
        F.col("id").isNull() |
        F.col("date").isNull() |
        F.col("store_nbr").isNull() |
        F.col("sales").isNull()
    ).count()

    assert null_count == 0

def test_silver_train_no_duplicate_ids():
    df = spark.table("databricks_gsales.silver.silver_train")

    duplicate_count = (
        df.groupBy("id")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    assert duplicate_count == 0

def test_sales_not_negative():
    df = spark.table("databricks_gsales.silver.silver_train")
    assert df.filter(F.col("sales") < 0).count() == 0

def test_store_number_unique():
    df = spark.table("databricks_gsales.silver.silver_stores")

    duplicates = (
        df.groupBy("store_nbr")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    assert duplicates == 0

def test_gold_sales_summary_not_empty():
    df = spark.table("databricks_gsales.gold.gold_sales_summary")
    assert df.count() > 0

def test_gold_total_sales_positive():
    df = spark.table("databricks_gsales.gold.gold_sales_summary")
    assert df.select("total_sales").first()[0] > 0

def test_all_gold_tables_not_empty():
    tables = [
        "gold_daily_sales",
        "gold_store_performance",
        "gold_family_sales",
        "gold_promotion_analysis",
        "gold_monthly_sales",
        "gold_sales_summary"
    ]

    for table in tables:
        df = spark.table(f"databricks_gsales.gold.{table}")
        assert df.count() > 0, f"{table} is empty"
"""

with open("/tmp/test_data_quality.py", "w") as f:
    f.write(test_code)

print("PyTest file created successfully")

PyTest file created successfully


In [0]:
import pytest

result = pytest.main([
    "-v",
    "/tmp/test_data_quality.py"
])

if result != 0:
    raise Exception("DATA QUALITY TESTS FAILED")

print("ALL DATA QUALITY TESTS PASSED")

============================= test session starts ==============================
platform linux -- Python 3.12.3, pytest-8.3.5, pluggy-1.5.0 -- /local_disk0/.ephemeral_nfs/envs/pythonEnv-11830c2c-9c33-4f5a-8d43-9a0feabf0388/bin/python
cachedir: .pytest_cache
rootdir: /tmp
plugins: langsmith-0.6.1, anyio-4.7.0
collecting ... collected 8 items

../../../tmp/test_data_quality.py::test_silver_train_not_empty PASSED    [ 12%]
../../../tmp/test_data_quality.py::test_silver_train_no_nulls PASSED     [ 25%]
../../../tmp/test_data_quality.py::test_silver_train_no_duplicate_ids PASSED [ 37%]
../../../tmp/test_data_quality.py::test_sales_not_negative PASSED        [ 50%]
../../../tmp/test_data_quality.py::test_store_number_unique PASSED       [ 62%]
../../../tmp/test_data_quality.py::test_gold_sales_summary_not_empty PASSED [ 75%]
../../../tmp/test_data_quality.py::test_gold_total_sales_positive PASSED [ 87%]
../../../tmp/test_data_quality.py::test_all_gold_tables_not_empty PASSED [100%]

=======